In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]
date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
volumes_all = np.empty((T, N))
prices_all = np.empty((T, N))
dfs_new = []
for i, df in enumerate(dfs):
    df = df[::-1]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")  # Handles str/NaN/object
    returns = df["Price"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)
    volumes_all[:, i] = df["Vol."].apply(correct_volume).values[1:]
    prices_all[:,i] = df["Price"].values[1:]
    dfs_new.append(df)

all_dates = dfs[0]["Date"].values[1:]
volumes_all = np.nan_to_num(volumes_all, nan=0.0)

In [2]:
import tensorflow as tf
import numpy as np

def sigmoid(x):
    return 1/(1+np.exp(-x))

def normalize(x):
    return (x - tf.reduce_mean(x)) / (tf.math.reduce_std(x) + 1e-8)

def rank_transform(x):
    ranks = tf.argsort(tf.argsort(x))
    return tf.cast(ranks, tf.float32) / tf.cast(tf.size(x), tf.float32)

def risk_parity_weights(train_data):
    # Compute volatilities for each asset
    volatilities = np.std(train_data, axis=0)  # (N,)
    inverse_volatilities = 1.0 / (volatilities + 1e-8)  # Avoid division by zero

    # Normalize to get weights
    weights = inverse_volatilities / np.sum(inverse_volatilities)
    return weights

C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 

In [16]:
import tensorflow as tf

def max_drawdown(prices, weights):
    # prices (T,N)
    # weights (N,)
    T, N = prices.shape
    portfolio_values = prices @ weights # (T,)
    highest_peak = -1
    worst_pct = 0.0
    for t in range(T):
        highest_peak = max(highest_peak, portfolio_values[t])
        worst_pct = min(worst_pct, portfolio_values[t] / highest_peak - 1)
    return worst_pct

def perform_validation(w, validation_data, validation_prices):
    # validation data is shape (T, N)
    
    #print('shape of validation_data_with_cash:', validation_data_with_cash.shape)
    #print('weights shape:', w.shape)

    # Reshape w to (8, 1) for matrix multiplication
    w = tf.reshape(w, (-1, 1))

    print('shapes:', w.shape, validation_data.shape)

    # perform metrics
    total_return = validation_data @ w  # shape (T, 1)
    total_return = total_return.numpy().flatten()  # convert to (T,)

    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return < 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)

    md = max_drawdown(validation_prices, w.numpy())

    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
    

In [19]:
def rolling_window(window_size, stepsize, validation_size, returns_data, volume_data):
    # data is (T,N)

    sharpes_w_opt= []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_eqw = []

    mds_w_opt = []
    mds_w_eqw = []


    T, N = returns_data.shape
    weights_opt = []

    for idx in range(0, T - window_size - validation_size, stepsize):
        train_data_returns = returns_data[idx:idx+window_size]
        val_data_returns = returns_data[idx+window_size:idx+window_size+validation_size]
        train_data_volume = volume_data[idx:idx+window_size]
        val_data_prices = prices_all[idx+window_size:idx+window_size+validation_size]
        #val_data_volume = volume_data[idx+window_size:idx+window_size+validation_size]
        #cov = np.cov(train_data_returns, rowvar=False)

        w = risk_parity_weights(train_data_returns)
        weights_opt.append(w)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(w, val_data_returns, val_data_prices)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)
        mds_w_opt.append(md)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return, md
        ) = perform_validation(np.ones(N) / N, val_data_returns, val_data_prices)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)
        mds_w_eqw.append(md)

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_opt)],
        "Highest maximum drawdowns": [np.min(mds_w_eqw), np.min(mds_w_opt)],
        "Mean maximum drawdowns": [np.mean(mds_w_eqw), np.mean(mds_w_opt)]
    }, index=["Equal weights", "Return + CVaR"])

    weights_opt = np.array(weights_opt)

    return df, weights_opt

In [21]:
df, weights_opt = rolling_window(
    window_size=252*2,
    stepsize=50,
    validation_size=252,
    returns_data=returns_all,
    volume_data=volumes_all,
)

df

shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (252, 7)
shapes: (7, 1) (

,Sharpes,Sortinos,Vars,Cvars,MeanReturn,Highest maximum drawdowns,Mean maximum drawdowns
Equal weights,0.922227,1.268908,-0.008891,-0.015423,0.000426,-0.281273,-0.127660
Return + CVaR,0.991049,1.360491,-0.008282,-0.014383,0.000429,-0.279578,-0.126928
